## 0. Imports & Chargement

In [85]:
import pandas as pd
import numpy as np
import re
import os
import json
import warnings
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

DATA_PATH      = '../data/raw/'
PROCESSED_PATH = '../data/processed/'
os.makedirs(PROCESSED_PATH, exist_ok=True)

df = pd.read_csv(PROCESSED_PATH + 'ds2_eda.csv')

print(f'Dataset chargé : {len(df):,} tweets')
print(f'Colonnes : {list(df.columns)}')
df.head(3)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Dataset chargé : 11,520 tweets
Colonnes : ['label_raw', 'text', 'label', 'label_name', 'word_count', 'char_count', 'has_url', 'has_mention', 'has_hashtag', 'has_emoji', 'has_latin', 'has_numbers']


,label_raw,text,label,label_name,word_count,char_count,has_url,has_mention,has_hashtag,has_emoji,has_latin,has_numbers
0,pos,#مسابقة والجائزة 💰 / من أول من فتق لسانه بالعر...,1,Positif,18,107,False,False,True,False,False,False
1,neg,❥↓🌿🍥 ما لأبن أدم والفخر اوله نطفه وأخره جيفه و...,0,Négatif,22,107,False,False,True,False,False,False
2,pos,"✨ لاتحزن ودع القلق,يستجيب لك الكريم هو يأخرها ...",1,Positif,21,116,False,False,False,False,False,False


## 1. Nettoyage

In [86]:
def clean_arabic(text):
    text = str(text)
    # Supprimer URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Supprimer mentions @ et hashtags # et remplace _ par espace
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'_', ' ', text)
    # Supprimer emojis et symboles
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    # Suppression des virgules 
    text = re.sub(r'[،,؛;:.!?]', '', text)
    # Supprimer chiffres
    text = re.sub(r'\d+', '', text)
    # Supprimer espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text_clean'] = df['text'].apply(clean_arabic)

# Vérification
print('Avant :')
print(df['text'].iloc[6])
print('\nAprès :')
print(df['text_clean'].iloc[6])

df.head()

Avant :
بمناسبة فوز الهلال .. 💙 سحب على آيفون XR📱 رتويت وتابع - السحب بعد ساعة موثق بالفديو 💪

Après :
بمناسبة فوز الهلال سحب على آيفون رتويت وتابع السحب بعد ساعة موثق بالفديو


,label_raw,text,label,label_name,word_count,char_count,has_url,has_mention,has_hashtag,has_emoji,has_latin,has_numbers,text_clean
0,pos,#مسابقة والجائزة 💰 / من أول من فتق لسانه بالعر...,1,Positif,18,107,False,False,True,False,False,False,مسابقة والجائزة من أول من فتق لسانه بالعربية ؟...
1,neg,❥↓🌿🍥 ما لأبن أدم والفخر اوله نطفه وأخره جيفه و...,0,Négatif,22,107,False,False,True,False,False,False,ما لأبن أدم والفخر اوله نطفه وأخره جيفه ولا ير...
2,pos,"✨ لاتحزن ودع القلق,يستجيب لك الكريم هو يأخرها ...",1,Positif,21,116,False,False,False,False,False,False,لاتحزن ودع القلقيستجيب لك الكريم هو يأخرها لوق...
3,neg,بيي الله يستر طلعتي من الدوام سيارتي وايد نازل...,0,Négatif,12,59,False,False,False,True,False,False,بيي الله يستر طلعتي من الدوام سيارتي وايد نازل...
4,neg,تلاتين سنة بترقص .. الليلة رقصتنا أنا ببكي 😭 د...,0,Négatif,13,63,False,False,False,True,False,False,تلاتين سنة بترقص الليلة رقصتنا أنا ببكي دي حلا...


## 2. Normalisation Arabe

In [87]:
def normalize_arabic(text):
    # Normaliser les alef
    text = re.sub(r'[أإآ]', 'ا', text)
    # Normaliser teh marbuta
    text = re.sub(r'ة', 'ه', text)
    # Normaliser alef ma9soura
    text = re.sub(r'ى', 'ي', text)
    # Supprimer tatweel
    text = re.sub(r'ـ', '', text)
    # Supprimer tashkil (diacritiques)
    text = re.sub(r'[\u064B-\u065F]', '', text)
    return text

df['text_clean'] = df['text_clean'].apply(normalize_arabic)

# Vérification
print('Avant normalisation :')
print(df['text'].iloc[1])
print('\nAprès normalisation :')
print(df['text_clean'].iloc[1])


Avant normalisation :
❥↓🌿🍥 ما لأبن أدم والفخر اوله نطفه وأخره جيفه ولا يرزق نفسه ولا يدفع حتفه {قال الأمام علي ع }♡💙🕊 #أتعبتني 💔…

Après normalisation :
ما لابن ادم والفخر اوله نطفه واخره جيفه ولا يرزق نفسه ولا يدفع حتفه قال الامام علي ع اتعبتني


## 3. Tokenisation & Stopwords

In [88]:
negation_words = {
    'لا', 'ليس', 'ليست', 'ليسوا', 'لست', 'لسنا', 'لستم', 'لستن',
    'لم', 'لما', 'لن', 'ما', 'غير', 'بدون', 'بلا', 'أبدا', 'قط'
}

arabic_stopwords = set(normalize_arabic(w) for w in stopwords.words('arabic'))
arabic_stopwords = arabic_stopwords - negation_words

def tokenize_and_remove_stopwords(text):
    tokens = text.split()
    tokens = [t for t in tokens if t not in arabic_stopwords]
    return ' '.join(tokens)

df['text_clean'] = df['text_clean'].apply(tokenize_and_remove_stopwords)

before = len(df)
df = df[df['text_clean'].str.strip() != ''].reset_index(drop=True)
after = len(df)

print(f'Tweets supprimés : {before - after}')
print(f'Total restant    : {after:,}')

print('\nAvant :')
print(df['text'].iloc[106])
print('\nAprès :')
print(df['text_clean'].iloc[106])

Tweets supprimés : 30
Total restant    : 11,490

Avant :
لا جد امطار الرياض تخوف امس واحد كان بيموت 😥

Après :
لا جد امطار الرياض تخوف بيموت


## 4. Suppression Doublons & Tweets Corrompus

In [89]:
before = len(df)

df = df.drop_duplicates(subset='text_clean').reset_index(drop=True)
df = df[df['word_count'] <= 50].reset_index(drop=True)

after = len(df)

print(f'Avant     : {before:,} tweets')
print(f'Après     : {after:,} tweets')
print(f'Supprimés : {before - after:,}')

Avant     : 11,490 tweets
Après     : 8,622 tweets
Supprimés : 2,868


## 5. Vectorisation TF-IDF

In [90]:
tfidf = TfidfVectorizer(max_features=10000)
X_tfidf = tfidf.fit_transform(df['text_clean'])

print(f'Shape matrice TF-IDF : {X_tfidf.shape}')
print(f'Vocabulaire size     : {len(tfidf.vocabulary_)}')

# Sauvegarder le vectoriseur
with open(PROCESSED_PATH + 'tfidf_vectorizer_ds2.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print('Sauvegardé : tfidf_vectorizer_ds2.pkl')

Shape matrice TF-IDF : (8622, 10000)
Vocabulaire size     : 10000
Sauvegardé : tfidf_vectorizer_ds2.pkl


In [91]:
maxlen = int(np.percentile(df['word_count'], 95))
print(f'maxlen : {maxlen}')

maxlen : 24


## 6. Séquences & Padding

In [92]:
VOCAB_SIZE = 10000
MAXLEN     = int(np.percentile(df['word_count'], 95))

# Créer et entraîner le tokenizer
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(df['text_clean'])

# Convertir en séquences
sequences = tokenizer.texts_to_sequences(df['text_clean'])

# Padding
X = pad_sequences(sequences, maxlen=MAXLEN, padding='post', truncating='post')

print(f'Shape final : {X.shape}')
print(f'Tweet nettoyé   : {df["text_clean"].iloc[106]}')
print(f'Exemple séquence brute  : {sequences[106]}')
print(f'Exemple après padding   : {X[106]}')

# Sauvegarder le tokenizer
with open(PROCESSED_PATH + 'keras_tokenizer_ds2.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print('Sauvegardé : keras_tokenizer_ds2.pkl')

Shape final : (8622, 24)
Tweet nettoyé   : سلامه الجميل سلامه مي يارب ماتشوفي شر حبيبتي الله يشفيكي ويعافيكي مرض ويحفظ
Exemple séquence brute  : [1143, 252, 1143, 2544, 9, 9138, 767, 559, 2, 5112, 9139, 858, 9140]
Exemple après padding   : [1143  252 1143 2544    9 9138  767  559    2 5112 9139  858 9140    0
    0    0    0    0    0    0    0    0    0    0]
Sauvegardé : keras_tokenizer_ds2.pkl


## 7. Split Train / Val / Test

In [93]:
from sklearn.model_selection import train_test_split

y = df['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)
print(f'Train : {X_train.shape}')
print(f'Val   : {X_val.shape}')
print(f'Test  : {X_test.shape}')

np.save(PROCESSED_PATH + 'X_train_pad_ds2.npy', X_train)
np.save(PROCESSED_PATH + 'X_val_pad_ds2.npy',   X_val)
np.save(PROCESSED_PATH + 'X_test_pad_ds2.npy',  X_test)
np.save(PROCESSED_PATH + 'y_train_ds2.npy',     y_train)
np.save(PROCESSED_PATH + 'y_val_ds2.npy',       y_val)
np.save(PROCESSED_PATH + 'y_test_ds2.npy',      y_test)

print('Tous les fichiers sauvegardés ')

Train : (6035, 24)
Val   : (1293, 24)
Test  : (1294, 24)
Tous les fichiers sauvegardés 


In [94]:
df[['text_clean', 'label']].to_csv(PROCESSED_PATH + 'ds2_preprocessed.csv', index=False)
print('Sauvegardé : ds2_preprocessed.csv ✅')

Sauvegardé : ds2_preprocessed.csv ✅


## 8. Résumé & Sauvegarde des Paramètres

In [95]:
params = {
    'vocab_size'   : VOCAB_SIZE,
    'maxlen'       : MAXLEN,
    'train_size'   : X_train.shape[0],
    'val_size'     : X_val.shape[0],
    'test_size'    : X_test.shape[0],
    'total_tweets' : len(df),
    'num_classes'  : 2
}

with open(PROCESSED_PATH + 'ds2_params.json', 'w') as f:
    json.dump(params, f, indent=4)

print('=' * 50)
print('   RÉSUMÉ SPRINT 2 — PREPROCESSING DS2')
print('=' * 50)
for k, v in params.items():
    print(f'  {k:<20} : {v}')
print('=' * 50)
print('Sauvegardé : ds2_params.json ✅')

   RÉSUMÉ SPRINT 2 — PREPROCESSING DS2
  vocab_size           : 10000
  maxlen               : 24
  train_size           : 6035
  val_size             : 1293
  test_size            : 1294
  total_tweets         : 8622
  num_classes          : 2
Sauvegardé : ds2_params.json ✅
